タイタニック号の生存者を予測するプログラム

決定木回帰モデルを用いる。

ローカルパスの設定


In [1]:
import os

# ローカルパスの設定
raw_dir     = '../data/raw/'
split_dir   = '../data/split_from_train/'
missing_dir = '../data/missing_store/'

# 出力ディレクトリの作成
os.makedirs(split_dir,   exist_ok=True)
os.makedirs(missing_dir, exist_ok=True)

print("パス設定完了")


パス設定完了


titanic dataを読み込む

- `data/raw/train.csv` ：学習・評価に使うデータ（Survived ラベル付き）
- `data/raw/test.csv` ：評価には使わない。欠損値補完の参照用として読み込み、`data/missing_store/` に保管する


In [2]:
import numpy as np
import pandas as pd

#
# 学習データの読み込み (data/raw/train.csv)
#
df = pd.read_csv(raw_dir + 'train.csv')

#
# テストデータの読み込み。
# READMEの方針: 評価には使わず、欠損値補完の参照用として data/missing_store/ に保管する
#
test_df = pd.read_csv(raw_dir + 'test.csv')
test_df.to_csv(missing_dir + 'test.csv', index=False)

print(f"学習データ: {df.shape}")
print(f"参照用テストデータ: {test_df.shape}")


学習データ: (891, 12)
参照用テストデータ: (418, 11)


dataの欠損値を補完する必要があります。

titanic dataには欠損値がたくさんあるので、何らかの補完をしておかないと学習ができないから。



In [3]:
#
# 覚えていないので、dfのカラムを確認しておく
#
print(df.columns)


Index(['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp',
       'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked'],
      dtype='str')


まず、カラムごとの欠損値がどのくらいあるのか確認しておく。

In [4]:
print("titanic data の欠損値")

print(df.isnull().sum())

# print("Embarked の欠損値=", df.Embarked.isnull().sum())

titanic data の欠損値
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64


欠損値を埋める。欠損値があるのはAge, Cabin, Embarkedの３つ。

Cabinは使えるとすごそうだが、ほとんど無いため、せめてAgeとEmbarkedは埋めたい。

年齢は中央値とする。

Cabinデータは学習には使わない。

Embarked（乗り込んだ港）は適当に埋める

In [5]:
#
# 年齢の平均値と中央値を確認するときにはコメント外してちょんまげ
#
# print(df.Age.mean())
# print(df.Age.median())

# 学習データとテストデータを連結する
all_df = pd.concat([df, test_df], ignore_index=True)
# print(fill_df)

#
# 学習データとテストデータが連結されていることを確認。
# 中央値は全体の値から計算したいのでこの手法をとる
#
print(all_df.shape)

#
# 念の為、dfをfill_dfにコピー
#
fill_df = df.copy()

#
# 年齢の中央値を全てのデータから計算
#
age_median=all_df.Age.median()

#
# 年齢の欠損値を、計算しておいた中央値で補完する
#
fill_df.Age = fill_df.Age.fillna(age_median)

#
# Embarked 欠損値の補完：どちらもS（サザンプトン港）としてしまう。
#
fill_df.Embarked = fill_df.Embarked.fillna('S')

#
# 乗船した港の欠損値を再度確認する
#
print("欠損値の確認")
print(fill_df.isnull().sum())
# fill_df.head()

(1309, 12)
欠損値の確認
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age              0
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         0
dtype: int64


データを学習データとテストデータに分割する。

fill_dfのデータを分割して学習データx_train, y_train, テストデータx_test, y_testを作成する。

EmbarkedとSexをホットエンコードに変換する


In [6]:
from sklearn.model_selection import train_test_split

#
# y_trainにSurvivedを挿入する
#
y_all = fill_df["Survived"]

#
# 使わないカラムを除く（Survived, Name, Ticket, Cabin）
#
x_all = fill_df.drop(columns=['Survived', 'Name', 'Ticket', 'Cabin'])

#
# Sex, Embarkedをone-hotエンコードに変換する
#
x_all = pd.get_dummies(x_all, columns=['Sex', 'Embarked'], prefix='HE')

#
# x_train / x_test を 8:2 に分割する（random_state で再現性を確保）
#
x_train, x_test, y_train, y_test = train_test_split(
    x_all, y_all, test_size=0.2, random_state=42)

#
# 分割したデータを data/split_from_train/ に保存する（READMEの方針）
#
train_split_df = x_train.copy()
train_split_df.insert(0, 'Survived', y_train.values)
train_split_df.to_csv(split_dir + 'train_split.csv', index=False)

valid_split_df = x_test.copy()
valid_split_df.insert(0, 'Survived', y_test.values)
valid_split_df.to_csv(split_dir + 'valid_split.csv', index=False)

print(f"学習用: {x_train.shape}, 評価用: {x_test.shape}")
print(f"分割データを {split_dir} に保存しました")
print(x_train.head())


学習用: (712, 11), 評価用: (179, 11)
分割データを ../data/split_from_train/ に保存しました
     PassengerId  Pclass   Age  SibSp  Parch     Fare  HE_female  HE_male  \
331          332       1  45.5      0      0  28.5000      False     True   
733          734       2  23.0      0      0  13.0000      False     True   
382          383       3  32.0      0      0   7.9250      False     True   
704          705       3  26.0      1      0   7.8542      False     True   
813          814       3   6.0      4      2  31.2750       True    False   

      HE_C   HE_Q  HE_S  
331  False  False  True  
733  False  False  True  
382  False  False  True  
704  False  False  True  
813  False  False  True  


決定木分類を学習する。

In [7]:
# ライブラリのインポート
from sklearn import tree
#
# 決定木モデルを作る
#
model = tree.DecisionTreeClassifier()

#
# 決定木モデルを学習
#
model.fit(x_train, y_train)

#
# 作成した決定木モデルを使った予測をおこなう。
#
y_pred = model.predict(x_test)

#
# 予測結果を表示
#
print(y_pred)

[0 0 0 1 1 1 1 0 1 1 0 0 0 0 0 1 1 0 0 0 0 1 0 0 0 1 0 1 0 1 0 0 0 1 0 0 1
 1 1 0 0 1 0 0 1 0 0 0 0 1 1 1 0 0 0 1 0 1 0 1 0 1 1 1 0 1 0 1 0 1 1 1 0 1
 0 0 1 1 1 1 0 1 1 0 0 0 1 1 0 0 0 0 1 0 0 0 0 0 1 0 0 0 1 0 0 0 1 0 1 0 1
 0 1 0 0 0 0 0 1 0 0 1 1 1 0 0 1 0 0 0 1 0 0 0 1 0 1 1 0 0 0 1 0 0 0 1 1 1
 1 0 0 0 0 0 0 0 1 1 1 1 0 0 0 1 0 0 0 1 0 0 0 1 0 0 0 0 0 1 1]


予測結果y_predと正しい結果y_testを比較して、認識率を計算する

In [8]:
from sklearn.metrics import accuracy_score

#
# 予測結果y_predと正しい結果y_testを比較して、認識率を計算する
#
accuracy = accuracy_score(y_test, y_pred)

#
# 推定率を表示
#
print("推定率=", accuracy)


推定率= 0.770949720670391
